<a href="https://colab.research.google.com/github/reedlm11/INFO648/blob/main/_INFO_648_Summer_26_Lesson_7_Regression_HW_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas

In [20]:
import pandas as pd
df=pd.read_csv('saas_deals.csv')

In [21]:
# Print the shape of the DataFrame
print(f"Shape of the DataFrame: {df.shape}")

Shape of the DataFrame: (300, 8)


In [22]:
# Display the first few rows of the DataFrame
df.head()

,deal_id,company_employees,sales_calls,pipeline_days,industry,plan_tier,region,acv_usd
0,D-0209,32,12,170,Retail,Enterprise,APAC,113043
1,D-0189,1631,17,119,Healthcare,Pro,APAC,75461
2,D-0013,855,24,71,Manufacturing,Enterprise,EMEA,129664
3,D-0222,322,20,19,Retail,Basic,NorthAmerica,10255
4,D-0240,30,28,99,Retail,Pro,APAC,56544


In [23]:
# Display descriptive statistics, including categorical columns
display(df.describe(include='all'))

,deal_id,company_employees,sales_calls,pipeline_days,industry,plan_tier,region,acv_usd
count,300,300.000000,300.000000,300.000000,300,300,300,300.000000
unique,300,NaN,NaN,NaN,5,3,3,NaN
top,D-0173,NaN,NaN,NaN,Tech,Pro,NorthAmerica,NaN
freq,1,NaN,NaN,NaN,80,144,170,NaN
mean,NaN,1177.373333,15.923333,92.843333,NaN,NaN,NaN,69109.313333
std,NaN,1717.906689,8.889368,48.290566,NaN,NaN,NaN,34415.112768
min,NaN,16.000000,1.000000,5.000000,NaN,NaN,NaN,5000.000000
25%,NaN,62.000000,8.000000,55.500000,NaN,NaN,NaN,43925.500000
50%,NaN,303.500000,17.000000,93.000000,NaN,NaN,NaN,60715.500000
75%,NaN,1675.250000,24.000000,130.500000,NaN,NaN,NaN,90609.750000


In [24]:
df.describe()

,company_employees,sales_calls,pipeline_days,acv_usd
count,300.000000,300.000000,300.000000,300.000000
mean,1177.373333,15.923333,92.843333,69109.313333
std,1717.906689,8.889368,48.290566,34415.112768
min,16.000000,1.000000,5.000000,5000.000000
25%,62.000000,8.000000,55.500000,43925.500000
50%,303.500000,17.000000,93.000000,60715.500000
75%,1675.250000,24.000000,130.500000,90609.750000
max,7626.000000,30.000000,180.000000,167606.000000


In [25]:
model_df=df.copy()

### Column Information

Based on our `model_df` (after dropping `deal_id`):

**Numeric Features**:
*   `company_employees`
*   `sales_calls`
*   `pipeline_days`

**Categorical Features**:
*   `industry`
*   `plan_tier`
*   `region`

**Target Variable**:
*   `acv_usd` (Annual Contract Value in USD, which we are aiming to predict).

In [26]:
# Define features (X) and target (y)
X = model_df.drop('acv_usd', axis=1)
y = model_df['acv_usd']

# Define lists for numeric and categorical features for clarity
numeric_features = ['company_employees', 'sales_calls', 'pipeline_days']
categorical_features = ['industry', 'plan_tier', 'region']

print("Features (X) head:")
display(X.head())
print("\nTarget (y) head:")
display(y.head())

Features (X) head:


,deal_id,company_employees,sales_calls,pipeline_days,industry,plan_tier,region
0,D-0209,32,12,170,Retail,Enterprise,APAC
1,D-0189,1631,17,119,Healthcare,Pro,APAC
2,D-0013,855,24,71,Manufacturing,Enterprise,EMEA
3,D-0222,322,20,19,Retail,Basic,NorthAmerica
4,D-0240,30,28,99,Retail,Pro,APAC



Target (y) head:


,acv_usd
0,113043
1,75461
2,129664
3,10255
4,56544


In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=0
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (225, 7)
X_test shape: (75, 7)
y_train shape: (225,)
y_test shape: (75,)


In [28]:
X_train

,deal_id,company_employees,sales_calls,pipeline_days,industry,plan_tier,region
259,D-0029,40,16,78,Retail,Pro,APAC
37,D-0183,390,4,173,Retail,Enterprise,NorthAmerica
97,D-0055,1256,17,143,Manufacturing,Enterprise,NorthAmerica
191,D-0163,64,29,30,Tech,Basic,EMEA
135,D-0199,51,10,12,Healthcare,Enterprise,EMEA
...,...,...,...,...,...,...,...
251,D-0018,22,8,142,Finance,Basic,EMEA
192,D-0037,34,8,32,Healthcare,Enterprise,NorthAmerica
117,D-0281,20,3,112,Finance,Pro,NorthAmerica
47,D-0065,807,18,172,Finance,Enterprise,EMEA


In [29]:
X.head()

,deal_id,company_employees,sales_calls,pipeline_days,industry,plan_tier,region
0,D-0209,32,12,170,Retail,Enterprise,APAC
1,D-0189,1631,17,119,Healthcare,Pro,APAC
2,D-0013,855,24,71,Manufacturing,Enterprise,EMEA
3,D-0222,322,20,19,Retail,Basic,NorthAmerica
4,D-0240,30,28,99,Retail,Pro,APAC


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# numeric_features and categorical_features are already defined from previous steps

# Build a ColumnTransformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), numeric_features),
        ("cat", OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

# Wrap the ColumnTransformer and LinearRegression in a single Pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)



In [31]:
model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', MinMaxScaler(),
                                                  ['company_employees',
                                                   'sales_calls',
                                                   'pipeline_days']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['industry', 'plan_tier',
                                                   'region'])])),
                ('regressor', LinearRegression())])

## Part 4 – Fit and evaluate

In [33]:
# Fit the Pipeline on the training data only
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', MinMaxScaler(),
                                                  ['company_employees',
                                                   'sales_calls',
                                                   'pipeline_days']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['industry', 'plan_tier',
                                                   'region'])])),
                ('regressor', LinearRegression())])

In [34]:
# Predict on the test data
y_pred = model.predict(X_test)

In [35]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Report MAE and RMSE on the test set in dollars
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")

# Interpret the MAE in business terms
print(f"\nOn average, our model is off by about ${mae:.2f} per deal.")

Mean Absolute Error (MAE): $10626.68
Root Mean Squared Error (RMSE): $13444.11

On average, our model is off by about $10626.68 per deal.


### What Did Our Preprocessor Do?

In [36]:
import pandas as pd

# Just so we can see what the preprocessor did to our data
X_processed = model.named_steps['preprocessor'].transform(X_test)
df_preview = pd.DataFrame(X_processed, columns=model.named_steps['preprocessor'].get_feature_names_out())

# Use display() for a pretty, scrollable table in Colab
display(df_preview.head())

,num__company_employees,num__sales_calls,num__pipeline_days,cat__industry_Healthcare,cat__industry_Manufacturing,cat__industry_Retail,cat__industry_Tech,cat__plan_tier_Enterprise,cat__plan_tier_Pro,cat__region_EMEA,cat__region_NorthAmerica
0,0.148269,0.103448,0.908571,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
1,0.002421,0.103448,0.588571,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
2,0.269905,0.482759,0.200000,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.251958,0.310345,0.411429,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
4,0.073636,0.620690,0.634286,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


## Part 5 – Interpret the Coefficients

In [37]:
import pandas as pd

# Get feature names from the preprocessor
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

# Get coefficients from the LinearRegression model
coefficients = model.named_steps['regressor'].coef_

# Create a DataFrame to display coefficients
coefficients_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
display(coefficients_df.sort_values(by='Coefficient', ascending=False))

,Feature,Coefficient
7,cat__plan_tier_Enterprise,80206.876781
0,num__company_employees,36274.223787
1,num__sales_calls,19328.910813
8,cat__plan_tier_Pro,18461.952665
2,num__pipeline_days,8364.446257
9,cat__region_EMEA,4497.898150
3,cat__industry_Healthcare,-2364.040533
10,cat__region_NorthAmerica,-5801.315309
4,cat__industry_Manufacturing,-16830.969316
6,cat__industry_Tech,-19276.813094


### Identifying Dropped Baseline Categories

The `OneHotEncoder` with `drop='first'` removes one category per feature to avoid multicollinearity. The dropped category becomes the baseline against which other categories for that feature are compared.

To identify the dropped baselines, we can look at the original unique values of each categorical feature and compare them with the features generated by the one-hot encoder.

In [38]:
for cat_feature in categorical_features:
    original_categories = sorted(model_df[cat_feature].unique().tolist())
    encoded_categories = [col.replace(f'cat__{cat_feature}_', '') for col in feature_names if col.startswith(f'cat__{cat_feature}_')]

    # The dropped category is the one in original_categories but not in encoded_categories
    dropped_baseline = [cat for cat in original_categories if cat not in encoded_categories]

    print(f"For '{cat_feature}', the dropped baseline category is: {dropped_baseline[0] if dropped_baseline else 'N/A'}")


For 'industry', the dropped baseline category is: Finance
For 'plan_tier', the dropped baseline category is: Basic
For 'region', the dropped baseline category is: APAC


### Interpretation of Coefficients (for the VP of Sales)

Here are interpretations of selected coefficients:

In [41]:
# Numeric Feature Interpretation (e.g., sales_calls)
sales_calls_coeff = coefficients_df[coefficients_df['Feature'] == 'num__sales_calls']['Coefficient'].values[0]
print(f"*   **Sales Calls:** Each additional sales call is associated with about ${sales_calls_coeff:.2f} more in ACV, holding everything else constant.")

# Categorical Feature Interpretation (e.g., plan_tier_Pro)
# Assuming 'Basic' was the dropped baseline for 'plan_tier'
plan_tier_pro_coeff = coefficients_df[coefficients_df['Feature'] == 'cat__plan_tier_Pro']['Coefficient'].values[0]
print(f"*   **Plan Tier (Pro vs. Basic):** A deal with a 'Pro' plan tier is associated with an ACV that is about ${plan_tier_pro_coeff:.2f} higher compared to a 'Basic' plan tier, holding everything else constant.")

# Another Feature Interpretation (e.g., company_employees)
company_employees_coeff = coefficients_df[coefficients_df['Feature'] == 'num__company_employees']['Coefficient'].values[0]
print(f"*   **Company Employees:** For every one unit increase in the scaled 'company_employees' feature, the ACV is associated with about ${company_employees_coeff:.2f} more, holding everything else constant.")

*   **Sales Calls:** Each additional sales call is associated with about $19328.91 more in ACV, holding everything else constant.
*   **Plan Tier (Pro vs. Basic):** A deal with a 'Pro' plan tier is associated with an ACV that is about $18461.95 higher compared to a 'Basic' plan tier, holding everything else constant.
*   **Company Employees:** For every one unit increase in the scaled 'company_employees' feature, the ACV is associated with about $36274.22 more, holding everything else constant.


## Part 6 – Bonus: Try the Lasso

### 16. Add 3 made-up junk columns to the dataset.

In [42]:
import numpy as np

# Re-create model_df from original df to ensure a clean start for bonus part
model_df = df.copy()

# Add the three made-up junk columns
np.random.seed(42) # for reproducibility
model_df['lucky_number'] = np.random.randint(1, 101, size=len(model_df))
model_df['paperclips_sold'] = np.random.randint(0, 51, size=len(model_df))
model_df['office_temperature'] = np.random.randint(60, 91, size=len(model_df))

print("DataFrame with junk columns added:")
display(model_df.head())

DataFrame with junk columns added:


,deal_id,company_employees,sales_calls,pipeline_days,industry,plan_tier,region,acv_usd,lucky_number,paperclips_sold,office_temperature
0,D-0209,32,12,170,Retail,Enterprise,APAC,113043,52,15,76
1,D-0189,1631,17,119,Healthcare,Pro,APAC,75461,93,2,66
2,D-0013,855,24,71,Manufacturing,Enterprise,EMEA,129664,15,19,83
3,D-0222,322,20,19,Retail,Basic,NorthAmerica,10255,72,23,82
4,D-0240,30,28,99,Retail,Pro,APAC,56544,61,32,89


### Re-define Features (X) and Target (y) and Re-split Data

Since we modified `model_df`, we need to re-define `X` and `y` and then re-split the data into training and testing sets.

In [43]:
# Define features (X) and target (y) again with the new columns
X_lasso = model_df.drop('acv_usd', axis=1) # Keeping 'deal_id' for now
y_lasso = model_df['acv_usd']

# Re-split the data
from sklearn.model_selection import train_test_split
X_train_lasso, X_test_lasso, y_train_lasso, y_test_lasso = train_test_split(
    X_lasso, y_lasso,
    test_size=0.25,
    random_state=0
)

print(f"X_train_lasso shape: {X_train_lasso.shape}")
print(f"X_test_lasso shape: {X_test_lasso.shape}")

# Update numeric_features to include the new junk columns
numeric_features_lasso = numeric_features + ['lucky_number', 'paperclips_sold', 'office_temperature']
print(f"\nUpdated numeric features for Lasso: {numeric_features_lasso}")

X_train_lasso shape: (225, 10)
X_test_lasso shape: (75, 10)

Updated numeric features for Lasso: ['company_employees', 'sales_calls', 'pipeline_days', 'lucky_number', 'paperclips_sold', 'office_temperature']


### 17. Re-fit your Pipeline using `LassoCV(cv=5)`

Now, we'll replace `LinearRegression` with `LassoCV` and use the updated numeric features. `MinMaxScaler` is already part of the numeric preprocessing, which is essential for Lasso.

In [44]:
from sklearn.linear_model import LassoCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Build a ColumnTransformer for preprocessing, including the new numeric features
# Note: deal_id is dropped here as it's neither numeric nor categorical for the preprocessor
preprocessor_lasso = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), numeric_features_lasso),
        ("cat", OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ],
    remainder='drop' # Explicitly drop 'deal_id' and any other unhandled columns
)

# Wrap the ColumnTransformer and LassoCV in a single Pipeline
model_lasso = Pipeline(
    steps=[
        ("preprocessor", preprocessor_lasso),
        ("regressor", LassoCV(cv=5, random_state=0))
    ]
)

# Fit the Lasso Pipeline on the training data
model_lasso.fit(X_train_lasso, y_train_lasso)

print("LassoCV model fitted successfully!")

LassoCV model fitted successfully!


### 18. Report which features LassoCV keeps (nonzero coefficients) and which it drops (coefficients equal to zero).

Let's extract the coefficients from the fitted LassoCV model and see which features have been retained.

In [45]:
import pandas as pd

# Get feature names from the preprocessor
feature_names_lasso = model_lasso.named_steps['preprocessor'].get_feature_names_out()

# Get coefficients from the LassoCV model
coefficients_lasso = model_lasso.named_steps['regressor'].coef_

# Create a DataFrame to display coefficients
coefficients_df_lasso = pd.DataFrame({'Feature': feature_names_lasso, 'Coefficient': coefficients_lasso})

print("Features with non-zero coefficients (kept by LassoCV):")
display(coefficients_df_lasso[coefficients_df_lasso['Coefficient'] != 0].sort_values(by='Coefficient', ascending=False))

print("\nFeatures with zero coefficients (dropped by LassoCV):")
dropped_features_df = coefficients_df_lasso[coefficients_df_lasso['Coefficient'] == 0]
if not dropped_features_df.empty:
    display(dropped_features_df.sort_values(by='Feature'))
else:
    print("No features were dropped (all coefficients are non-zero).")

# Check if junk features were dropped
junk_features_names = [f'num__lucky_number', f'num__paperclips_sold', f'num__office_temperature']
junk_features_dropped = all(coeff == 0 for coeff in coefficients_df_lasso[coefficients_df_lasso['Feature'].isin(junk_features_names)]['Coefficient'])

print(f"\nDid LassoCV drop the junk features? {'Yes' if junk_features_dropped else 'No'}")

Features with non-zero coefficients (kept by LassoCV):


,Feature,Coefficient
10,cat__plan_tier_Enterprise,78737.621402
0,num__company_employees,34295.082188
1,num__sales_calls,17560.508323
11,cat__plan_tier_Pro,17496.837005
2,num__pipeline_days,7242.593680
12,cat__region_EMEA,3623.272362
5,num__office_temperature,-125.204720
13,cat__region_NorthAmerica,-5992.537954
7,cat__industry_Manufacturing,-14123.123854
9,cat__industry_Tech,-16982.944020



Features with zero coefficients (dropped by LassoCV):


,Feature,Coefficient
6,cat__industry_Healthcare,-0.0
3,num__lucky_number,0.0
4,num__paperclips_sold,0.0



Did LassoCV drop the junk features? No


### 19. Compare the test MAE of your Lasso model to the plain linear regression from Part 4.

Finally, let's evaluate the LassoCV model's performance and compare it to the initial `LinearRegression` model.

In [46]:
from sklearn.metrics import mean_absolute_error

# Predict on the test data using the Lasso model
y_pred_lasso = model_lasso.predict(X_test_lasso)

# Calculate MAE for Lasso model
mae_lasso = mean_absolute_error(y_test_lasso, y_pred_lasso)

print(f"LassoCV Model Mean Absolute Error (MAE): ${mae_lasso:.2f}")
print(f"Original Linear Regression Model MAE: ${mae:.2f}")

if mae_lasso < mae:
    print(f"The LassoCV model performs better than the original Linear Regression model by ${mae - mae_lasso:.2f} MAE.")
elif mae_lasso > mae:
    print(f"The LassoCV model performs worse than the original Linear Regression model by ${mae_lasso - mae:.2f} MAE.")
else:
    print("The LassoCV model performs similarly to the original Linear Regression model.")

LassoCV Model Mean Absolute Error (MAE): $10303.36
Original Linear Regression Model MAE: $10626.68
The LassoCV model performs better than the original Linear Regression model by $323.32 MAE.
